# Documentation and resources

**Big Query**
- Colab has an example notebook on BigQuery too.  File > Open notebook > Examples > Getting Started with BigQuery.
- Also a Big Query Snippets Example Notebook
- [BigQuery Documentation]( https://cloud.google.com/bigquery/docs )
- [Open Data Sets]( https://console.cloud.google.com/marketplace/browse?filter=solution-type:dataset )
- [Reddit - list of data sets available on BQ]( https://www.reddit.com/r/bigquery/wiki/datasets )

**Big Query Console**  
- [Google Cloud Console]( https://console.cloud.google.com )
- Make sure your project is selected
- Scroll down to BigQuery on the left menu
- [Setup and query instructions]( https://cloud.google.com/bigquery/docs/quickstarts/query-public-dataset-console )

**SQL**
- [Kaggle Intro to SQL]( https://www.kaggle.com/learn/intro-to-sql ) uses BigQuery
- [Kaggle Advanced SQL]( https://www.kaggle.com/learn/advanced-sql )

**BigQuery Public Data Sets**
- [Austin Crime]( https://console.cloud.google.com/marketplace/product/city-of-austin/austin-crime )


#  Linking BigQuery to Colab

## Getting started

**You will only need to do this part once.**

1. Use the [Cloud Resource Manager](https://console.cloud.google.com/cloud-resource-manager) to Create a Cloud Platform project if you do not already have one.

    - Create Project
    - Project Name

2. [Enable BigQuery APIs](https://console.cloud.google.com/flows/enableapi?apiid=bigquery) for the project

**Note:** You get 1 TB/month of free queries for open datasets
- Kaggle gives you 5 TB/month free



## Imports

In [1]:
import pandas as pd
import pandas_gbq
from google.cloud import bigquery
from google.colab import auth
from google.colab import syntax
from google.colab import userdata


In [2]:
%alias gcloud gcloud

### Provide your credentials

In [4]:
auth.authenticate_user()
print('Authenticated')

Authenticated


## Optional: Enable data table display

Colab includes the ``google.colab.data_table`` package that can be used to display large pandas dataframes as an interactive data table.
It can be enabled with:

In [5]:
%load_ext google.colab.data_table
# %unload_ext google.colab.data_table

If you would prefer to return to the classic Pandas dataframe display, you can disable this by running:

In [12]:
%unload_ext google.colab.data_table

## List projects



In order to query BigQuery, you will need to specify a project ID.  To get a list of project IDs associated with your account, run the following command.

In [6]:
gcloud projects list --sort-by=projectId

PROJECT_ID                   NAME                    PROJECT_NUMBER  ENVIRONMENT
cool-monolith-286222         Data Science            271608828771
data-science-project-321016  Data Science Project    540766130804
gen-lang-client-0664606651   Default Gemini Project  566992509144


To take a quick anonymous survey, run:
  $ gcloud survey



## Declare the Cloud project ID which will be used throughout this notebook

Create a secret in your vault named `bq_billing_project_id` and assign it the value of the project ID you want to use.

1. open the vault ( key icon on the far left )
1. click "Add new secret"
1. type the name ( `bq_billing_project_id` )
1. paste the value ( do NOT type enter )
1. click back in the notebook


Fetch the project ID from your vault.

In [7]:
project_id = userdata.get('bq_billing_project_id')
project_id

'cool-monolith-286222'

## Samples data set



The [GSOD table](https://console.cloud.google.com/bigquery?p=bigquery-public-data&d=samples&t=gsod&page=table) in the Samples data set contains weather information collected by NOAA, such as precipitation amounts and wind speeds from late 1929 to early 2010.


# Use BigQuery via magics



The `google.cloud.bigquery` library also includes a magic command which runs a query and either displays the result or saves it to a variable as a `DataFrame`.

In [8]:
# Display query output immediately

%%bigquery --project {project_id}
SELECT
  COUNT(1) as total_rows
FROM `bigquery-public-data.samples.gsod`


Query is running:   0%|          |

Downloading:   0%|          |

,total_rows
0,114420316


In [9]:
# Save output in a variable `df`

%%bigquery bq_public_data_df --project {project_id}
SELECT
  COUNT(1) as total_rows
FROM `bigquery-public-data.samples.gsod`


Query is running:   0%|          |

Downloading:   0%|          |

In [13]:
bq_public_data_df

,total_rows
0,114420316


In [11]:
f'{bq_public_data_df.iloc[0,0]:,.0f}'

'114,420,316'

# Use BigQuery through google-cloud-bigquery



See [BigQuery documentation](https://cloud.google.com/bigquery/docs) and [library reference documentation](https://googlecloudplatform.github.io/google-cloud-python/latest/bigquery/usage.html).


### Count total number of rows

In [14]:
query = '''
  SELECT
    COUNT(1) as total
  FROM `bigquery-public-data.samples.gsod`
  '''

with bigquery.Client(project=project_id) as client:
  bq_df = client.query( query ).to_dataframe()


In [15]:
bq_df.shape


(1, 1)

In [16]:
bq_df

,total
0,114420316


In [17]:
query = '''
  SELECT
    COUNT(1) as total
  FROM `bigquery-public-data.samples.gsod`
  '''

with bigquery.Client(project=project_id) as client:
  row_count = client.query( query ).to_dataframe()["total"][0]

print(f'Full dataset has {row_count:,.0f} rows')


Full dataset has 114,420,316 rows


## Sample approximately 2000 random rows

### Describe the sampled data

In [18]:
sample_count = 2000

query = f'''
  SELECT
    *
  FROM
    `bigquery-public-data.samples.gsod`
  WHERE RAND() < {sample_count}/{row_count}
'''

with bigquery.Client(project=project_id) as client:
  df = client.query( query ).to_dataframe()

df


,station_number,wban_number,year,month,day,mean_temp,num_mean_temp_samples,mean_dew_point,num_mean_dew_point_samples,mean_sealevel_pressure,...,min_temperature,min_temperature_explicit,total_precipitation,snow_depth,fog,rain,snow,hail,thunder,tornado
0,620100,99999,1960,5,31,70.000000,8,59.200001,8,1014.799988,...,NaN,<NA>,0.00,NaN,True,True,True,True,True,True
1,872570,99999,1964,12,29,64.599998,5,55.400002,5,NaN,...,NaN,<NA>,0.00,NaN,False,False,False,False,False,False
2,948510,99999,1973,4,14,62.599998,4,59.000000,4,1010.799988,...,NaN,<NA>,NaN,NaN,False,False,False,False,False,False
3,648600,99999,1976,2,29,90.900002,7,51.500000,7,1005.000000,...,NaN,<NA>,0.00,NaN,False,False,False,False,False,False
4,477150,99999,1977,12,5,45.900002,24,39.200001,24,NaN,...,NaN,<NA>,0.00,NaN,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2049,872440,99999,1992,4,27,64.099998,6,61.599998,6,1018.000000,...,NaN,<NA>,0.00,NaN,False,False,False,False,False,False
2050,712550,99999,1994,4,5,21.200001,22,3.300000,22,NaN,...,NaN,<NA>,0.19,NaN,False,False,False,False,False,False
2051,938000,99999,1995,2,18,58.700001,8,57.700001,8,1017.700012,...,NaN,<NA>,1.34,NaN,False,False,False,False,False,False
2052,937810,99999,2004,2,12,59.299999,8,47.799999,8,1010.400024,...,NaN,<NA>,0.00,NaN,False,False,False,False,False,False


In [19]:
df.index.size

2054

In [20]:
df.describe().transpose().astype({"count": int})

,count,mean,std,min,25%,50%,75%,max
station_number,2054,521854.823272,295797.054998,10080.0,269700.0,565595.0,725596.5,999999.0
wban_number,2054,89260.139727,27419.871604,91.0,99999.0,99999.0,99999.0,99999.0
year,2054,1987.187926,16.088126,1937.0,1977.0,1990.0,2000.0,2010.0
month,2054,6.575463,3.524032,1.0,3.0,7.0,10.0,12.0
day,2054,15.616358,8.881306,1.0,8.0,16.0,23.0,31.0
mean_temp,2054,51.679017,24.895163,-60.0,37.200001,55.900002,70.674997,105.699997
num_mean_temp_samples,2054,13.271178,7.938817,4.0,7.0,8.0,24.0,24.0
mean_dew_point,1963,40.891544,23.379995,-53.599998,27.849999,43.700001,57.299999,80.699997
num_mean_dew_point_samples,1963,13.219562,7.944647,4.0,7.0,8.0,24.0,24.0
mean_sealevel_pressure,1555,1015.192283,9.654001,938.400024,1009.700012,1014.799988,1020.400024,1056.699951


### View the first 10 rows

In [21]:
df.head(10)

,station_number,wban_number,year,month,day,mean_temp,num_mean_temp_samples,mean_dew_point,num_mean_dew_point_samples,mean_sealevel_pressure,...,min_temperature,min_temperature_explicit,total_precipitation,snow_depth,fog,rain,snow,hail,thunder,tornado
0,620100,99999,1960,5,31,70.000000,8,59.200001,8,1014.799988,...,NaN,<NA>,0.00,NaN,True,True,True,True,True,True
1,872570,99999,1964,12,29,64.599998,5,55.400002,5,NaN,...,NaN,<NA>,0.00,NaN,False,False,False,False,False,False
2,948510,99999,1973,4,14,62.599998,4,59.000000,4,1010.799988,...,NaN,<NA>,NaN,NaN,False,False,False,False,False,False
3,648600,99999,1976,2,29,90.900002,7,51.500000,7,1005.000000,...,NaN,<NA>,0.00,NaN,False,False,False,False,False,False
4,477150,99999,1977,12,5,45.900002,24,39.200001,24,NaN,...,NaN,<NA>,0.00,NaN,False,False,False,False,False,False
5,37460,99999,1977,2,22,44.799999,24,41.599998,24,994.799988,...,NaN,<NA>,0.12,NaN,False,False,False,False,False,False
6,784820,99999,1978,12,2,79.400002,6,68.300003,6,1016.099976,...,NaN,<NA>,0.00,NaN,False,False,False,False,False,False
7,129600,99999,1978,12,5,23.299999,7,12.700000,7,1024.699951,...,NaN,<NA>,0.00,NaN,False,False,False,False,False,False
8,151940,99999,1981,8,21,69.800003,5,57.599998,5,1009.900024,...,NaN,<NA>,0.00,NaN,False,False,False,False,False,False
9,724085,94732,1987,8,5,79.400002,22,72.000000,22,NaN,...,NaN,<NA>,0.00,NaN,True,True,True,True,True,True


In [22]:
df.isnull().sum()

,0
station_number,0
wban_number,0
year,0
month,0
day,0
mean_temp,0
num_mean_temp_samples,0
mean_dew_point,91
num_mean_dew_point_samples,91
mean_sealevel_pressure,499


In [23]:
# 10 highest total_precipitation samples
(
df
  .sort_values('total_precipitation', ascending=False)
  .head(10)
  [['station_number', 'year', 'month', 'day', 'total_precipitation']]
)


,station_number,year,month,day,total_precipitation
530,911610,1978,1,8,13.27
773,916290,1985,3,20,4.67
377,235140,1989,1,11,3.19
257,239210,1960,11,13,2.95
93,801120,1997,3,21,1.81
958,156550,1973,6,20,1.73
131,547360,1982,8,18,1.65
14,143700,1997,9,15,1.65
1614,860860,2004,5,24,1.54
1089,133760,1976,6,26,1.54


# Use BigQuery through pandas-gbq



The `pandas-gbq` library is a community led project by the pandas community. It covers basic functionality, such as writing a DataFrame to BigQuery and running a query, but as a third-party library it may not handle all BigQuery features or use cases.

[Pandas GBQ Documentation]( https://pandas-gbq.readthedocs.io/en/latest/ )

In [24]:
query = '''
  SELECT
    name, SUM(number) as count
  FROM
    bigquery-public-data.usa_names.usa_1910_2013
  WHERE
    state = 'TX'
  GROUP BY
    name
  ORDER BY
    count DESC
  LIMIT
    100
'''

df = pandas_gbq.read_gbq( query, project_id=project_id, )
df.head()


Downloading: 100%|██████████|


,name,count
0,James,272793
1,John,235139
2,Michael,225320
3,Robert,220399
4,David,219028
